# E1 — Kiểm chứng Profiler

Notebook đánh giá profiler dựa trên dữ liệu CSV được trích xuất từ các file JSON kết quả.

**Nguồn dữ liệu:**
- `results/e1_alpha_beta.csv` — E1.1: tham số truyền thông α/β
- `results/e1_tblock.csv` — E1.2: so sánh T_block isolated vs representative

**Chú ý:** `T_block_with_microbatches` được thêm vào JSON export gần đây. Các file JSON cũ chưa có field này — cần chạy lại experiment để có dữ liệu E1.2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

RESULTS_DIR = 'results'

df_ab = pd.read_csv(os.path.join(RESULTS_DIR, 'e1_alpha_beta.csv'))
df_tb = pd.read_csv(os.path.join(RESULTS_DIR, 'e1_tblock.csv'))

print('E1.1 rows:', len(df_ab))
print('E1.2 rows:', len(df_tb), '| with repr data:', df_tb['T_block_repr_ms'].notna().sum())

display(df_ab.head())

## E1.1 — Tham số truyền thông α/β

In [ ]:
# Aggregate statistics
def compute_stats(series):
    return pd.Series({
        'mean': series.mean(),
        'std': series.std(),
        'CV_%': series.std() / series.mean() * 100 if series.mean() != 0 else np.nan,
        'min': series.min(),
        'max': series.max(),
        'n': len(series),
    })

summary = df_ab.groupby(['connection']).agg({
    'alpha_us': compute_stats,
    'beta_ns_per_B': compute_stats,
})

print('=== E1.1 Summary by Connection Type ===\n')
display(summary)

In [ ]:
# Per-model summary table (matching guide.md format)
model_summary = df_ab.groupby(['model', 'connection']).agg(
    alpha_mean=('alpha_us', 'mean'),
    alpha_std=('alpha_us', 'std'),
    beta_mean=('beta_ns_per_B', 'mean'),
    beta_std=('beta_ns_per_B', 'std'),
    n=('alpha_us', 'count'),
).round({'alpha_mean': 2, 'alpha_std': 2, 'beta_mean': 4, 'beta_std': 4})

print('\n=== E1.1 Per-Model Breakdown ===\n')
display(model_summary)

In [ ]:
# Visualization: alpha distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for conn, color in zip(['intra', 'inter'], ['steelblue', 'coral']):
    sub = df_ab[df_ab['connection'] == conn]
    axes[0].hist(sub['alpha_us'], bins=15, alpha=0.6, label=f'{conn}-node', color=color, edgecolor='black')
    axes[1].hist(sub['beta_ns_per_B'], bins=15, alpha=0.6, label=f'{conn}-node', color=color, edgecolor='black')

axes[0].set_xlabel('alpha (us)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('E1.1: Latency alpha Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('beta (ns/B)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('E1.1: Bandwidth beta Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_ab.boxplot(column='alpha_us', by='connection', ax=axes[0])
axes[0].set_title('alpha by Connection Type')
axes[0].set_xlabel('Connection')
axes[0].set_ylabel('alpha (us)')

df_ab.boxplot(column='beta_ns_per_B', by='connection', ax=axes[1])
axes[1].set_title('beta by Connection Type')
axes[1].set_xlabel('Connection')
axes[1].set_ylabel('beta (ns/B)')

plt.suptitle('')  # remove default suptitle
plt.tight_layout()
plt.show()

## E1.2 — So sánh T_block_isolated và T_block_representative

In [ ]:
df_repr = df_tb[df_tb['T_block_repr_ms'].notna()].copy()

if len(df_repr) == 0:
    print('\n[E1.2] No T_block_with_microbatches data available yet.\n')
    print('This field was recently added to the JSON export.\n')
    print('Please re-run experiments to populate this data.\n')
else:
    df_repr['ratio'] = df_repr['T_block_repr_ms'] / df_repr['T_block_isolated_ms']
    print('\n=== E1.2 T_block_isolated vs Representative ===\n')
    display(df_repr[['model', 'world_size', 'microbatches', 'T_block_isolated_ms', 'T_block_repr_ms', 'ratio']])
    print(f'\nMean ratio (repr/isolated): {df_repr["ratio"].mean():.3f}')
    print(f'Range: [{df_repr["ratio"].min():.3f}, {df_repr["ratio"].max():.3f}]\n')
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(df_repr))
    width = 0.35
    ax.bar(x - width/2, df_repr['T_block_isolated_ms'], width, label='Isolated', color='steelblue')
    ax.bar(x + width/2, df_repr['T_block_repr_ms'], width, label='Representative', color='coral')
    ax.set_ylabel('Time (ms)')
    ax.set_title('E1.2: T_block_isolated vs T_block_representative')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{r.model}\nws={r.world_size} M={r.microbatches}' for r in df_repr.itertuples()], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(df_repr)), df_repr['ratio'], color='seagreen', edgecolor='black')
    ax.axhline(y=1.0, color='red', linestyle='--', label='ratio = 1.0')
    ax.set_ylabel('Ratio (repr / isolated)')
    ax.set_title('E1.2: Memory Pressure Ratio')
    ax.set_xticks(range(len(df_repr)))
    ax.set_xticklabels([f'{r.model} ws={r.world_size}' for r in df_repr.itertuples()], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Tổng kết E1

In [ ]:
print('\n=== E1 Profiler Validation Summary ===\n')

# E1.1 summary
print('[E1.1] Communication Parameters\n')
for conn in ['intra', 'inter']:
    sub = df_ab[df_ab['connection'] == conn]
    print(f'  {conn.upper()}-NODE (n={len(sub)}):\n')
    print(f"    alpha = {sub['alpha_us'].mean():.2f} +- {sub['alpha_us'].std():.2f} us  (CV = {sub['alpha_us'].std()/sub['alpha_us'].mean()*100:.1f}%)\n")
    print(f"    beta  = {sub['beta_ns_per_B'].mean():.4f} +- {sub['beta_ns_per_B'].std():.4f} ns/B  (CV = {sub['beta_ns_per_B'].std()/sub['beta_ns_per_B'].mean()*100:.1f}%)\n")

# Compare intra vs inter
intra_alpha = df_ab[df_ab['connection']=='intra']['alpha_us']
inter_alpha = df_ab[df_ab['connection']=='inter']['alpha_us']
intra_beta = df_ab[df_ab['connection']=='intra']['beta_ns_per_B']
inter_beta = df_ab[df_ab['connection']=='inter']['beta_ns_per_B']

print(f'  Intra/Inter ratio alpha: {intra_alpha.mean()/inter_alpha.mean():.2f}x\n')
print(f'  Intra/Inter ratio beta:  {intra_beta.mean()/inter_beta.mean():.2f}x\n')

# E1.2 summary
if len(df_repr) > 0:
    print('[E1.2] Representative Block Profiling\n')
    print(f"  Mean ratio T_repr / T_iso = {df_repr['ratio'].mean():.3f}\n")
    print('  Confirms memory pressure effect: representative >= isolated\n')
else:
    print('[E1.2] Representative Block Profiling\n')
    print('  Data not yet available. Re-run experiments with updated export.\n')